In [22]:
import datetime
import os
from glob import glob
import netCDF4 as nc
import numpy as np
import pandas as pd

### Load data

In [23]:
data_path = 'ME4OH_EN411_OFAM3/data/en4.1.1/1993-2014/'
files = [f for f in glob(os.path.join(data_path, "*.nc"))]

In [24]:
def load_and_filter_ohc(filepath):
    """Load data file, extract variables, filter for layer 1 OHC"""
    file_data = nc.Dataset(filepath,'r')

    # Extract variables
    lat = file_data["ts_lat"][:].data  # Latitude
    long = file_data["ts_lon"][:].data  # Longitude
    time = file_data["en4_ymd"][:].data  # Year , month , day
    mask = file_data["dohc_mask_by_en4_maxdepth"][:].data # Valid profile mask
    dohc1 = file_data["dohc"][:, 0].data  # Layer 1 OHC anomaly target (NOTE: index 0 in python, index 1 in R)
    ssh = file_data["eta_t"][:].data  # Sea surface height

    # Filter to valid OHC
    # Only valid Layer-1 profiles included according to mask and remove NaNs
    idx = [i for i in range(len(mask)) if mask[i, 0] == 1]
    filtered_dohc1 = np.array([dohc1[i] for i in idx])
    filtered_lat = np.array([lat[i] for i in idx])
    filtered_long = np.array([long[i] for i in idx])
    filtered_date = np.array([datetime.date(int(time[i, 0]), int(time[i, 1]), int(time[i, 2])) for i in idx])
    filtered_ssh = np.array([ssh[i] for i in idx])

    # Make df
    ohc_df = pd.DataFrame({
        "Date": filtered_date,
        "Latitude": filtered_lat,
        "Longitude": filtered_long,
        "OHC": filtered_dohc1,
        "SSH": filtered_ssh
    })
    ohc_df = ohc_df.dropna()
    
    return ohc_df
    

In [25]:
# Temporarily filter to only 10 files
# files = files[0:10]

In [26]:
ohc_df = load_and_filter_ohc(files[0])
for path in files[1:]:
    file_df = load_and_filter_ohc(path)
    ohc_df = pd.concat([ohc_df, file_df])

ohc_df

,Date,Latitude,Longitude,OHC,SSH
0,1993-09-12,-65.050003,236.649994,0.343876,-1.691031
1,1993-09-10,-64.050003,221.550003,0.344592,-1.746879
2,1993-09-15,-63.549999,244.350006,0.345307,-1.345561
3,1993-09-11,-63.349998,278.549988,0.345918,-1.107517
4,1993-09-12,-62.849998,219.149994,0.344311,-1.786248
...,...,...,...,...,...
7445,1995-06-28,73.650002,19.150000,0.347956,-0.737022
7446,1995-06-29,73.750000,13.250000,0.348566,-0.766320
7447,1995-06-05,73.750000,13.350000,0.348498,-0.795618
7448,1995-06-29,73.750000,13.350000,0.348606,-0.762963


# EDA

- Amount of data points across time
- Amount of data across space
- Plot OHC vs lat long for a single month?